# Disagreement-Gated Teachers

**One fixed experiment:** Vary the teacher contribution from 29% to 31% according to the percentile rank of disagreement among the four public members. No labels enter the gate.

This notebook writes one `submission.csv`. It does not train a model or submit automatically. Coefficients are fixed before this candidate's leaderboard submission. Ranking scores are not calibrated probabilities.

## Why this experiment

The fresh public leaders [Nina h-blend (4), V8](https://www.kaggle.com/code/nina2025/ps-s6e10-h-blend-4?scriptVersionId=360200132) and [h-blend (5), V8](https://www.kaggle.com/code/nina2025/ps-s6e10-h-blend-5?scriptVersionId=360199897) both score **0.96205**. They apply the same within-row ordering weights to four public predictions, with a 30% additional prediction from different teacher/stack views. We implement that rule with NumPy and first reproduce both public outputs' **entire rank vectors exactly**. This check runs below before our single change.

This is experiment **8/9** in our study. The three earlier [old-anchor trials](https://www.kaggle.com/code/kirilltym/s6e10-nine-rank-ensemble-experiments?scriptVersionId=360178725) scored 0.96204, 0.96203 and 0.96204. After finding the new 0.96205 leaders, the remaining six experiments were rebased and published separately. A better public digit would be a public-board result; the anchor OOFs are unavailable, so no independent CV improvement is claimed.

## Credit and pinned inputs

- **F.A.Nina** — the two h-blend notebooks above, the row-order combination rule, and the [public input dataset](https://www.kaggle.com/datasets/nina2025/ps-s6e10-7).
- **Chris Deotte and upstream authors** — [Stacking TFMs with Public OOF](https://www.kaggle.com/code/cdeotte/stacking-tfms-with-public-oof), represented by the 0.96176 input.
- **Den Pugovkin and upstream authors** — [Passenger Context + Foundation Ensemble](https://www.kaggle.com/code/denpugovkin/s6e10-passenger-context-foundation-ensemble), represented by the 0.96183 input.
- The four 0.96201–0.96204 input files are community prediction files rehosted by Nina. The current 0.96204 file is byte-identical to our [earlier Agentic/Surface rank consensus](https://www.kaggle.com/code/kirilltym/s6e10-nine-rank-ensemble-experiments?scriptVersionId=360178725), which itself credits Hanif Noer Rofiq and samanyu1808. Rehosting does not make the underlying models ours.

All six source files and both reproduced rank vectors are hash-pinned. The members' model work belongs to the credited authors; our contribution is this single transformation, its checks and the measured result attached to this notebook.


In [ ]:
from pathlib import Path
import hashlib, json, os
import numpy as np
import pandas as pd
from scipy.stats import rankdata
from scipy.special import logit

INPUT = Path(os.environ.get('AIRLINE_INPUT_DIR', '/kaggle/input'))
OUTPUT = Path(os.environ.get('AIRLINE_OUTPUT_DIR', '/kaggle/working'))
OUTPUT.mkdir(parents=True, exist_ok=True)
N = 299_844
SOURCES = {'0.96201.csv': {'sha256': '8883773ff30ba9a71581657b9daf881ac45652031b6e3ee565844dbcff35797a', 'rows': 299844}, '0.96202.csv': {'sha256': '4765dbd769768cb9a7cbe030b9ecfd83a12b4fab25529d135cc6fef6fbc6e028', 'rows': 299844}, '0.96203.csv': {'sha256': 'b31119ef653e50b71452187174d6c85d7929550c23b4b020797f19f5f132112d', 'rows': 299844}, '0.96204.csv': {'sha256': 'c7d3b9e4c1c377c74e0f75b475c50319413a264fc4401c0a00fe689a99542873', 'rows': 299844}, '0.96176.csv': {'sha256': '9b5dc8e667fdecb388e56afd4772ecfb386cfb783f48b4d2270e747ebb371471', 'rows': 299844}, '0.96183.csv': {'sha256': '9f499e3b36232a0969dc50c26968946935808503a1f6abb562b91452251211db', 'rows': 299844}}
REFERENCE_RANK_HASHES = {'h4': 'b7f3f22764226369221f20b36e8e55d356a22bbe0f3a9be12a7affc17c0455c1', 'h5': 'f74910809b2b21a3ba8cee3b855e746cc4198ee5c533803167f043661c3f2daa'}
RECIPE = 'disagreement_gate'

def sha(p):
    return hashlib.sha256(p.read_bytes()).hexdigest()

def find(filename, slug):
    hits = [p for p in INPUT.rglob(filename) if slug in p.parts]
    assert len(hits) == 1, (filename, slug, [str(p) for p in hits])
    return hits[0]

def ranks(x):
    x = np.asarray(x, dtype=np.float64)
    assert x.shape == (N,) and np.isfinite(x).all()
    return rankdata(x, method='average')

def R(x):
    return (ranks(x) - 0.5) / N

sample = pd.read_csv(find('sample_submission.csv', 'playground-series-s6e10'))
test_ids = pd.read_csv(find('test.csv', 'playground-series-s6e10'), usecols=['id']).id.to_numpy()
ids = sample.id.to_numpy()
assert len(ids) == N and sample.id.is_unique
assert np.array_equal(ids, test_ids)
values = {}
for name, info in SOURCES.items():
    path = find(name, 'ps-s6e10-7')
    assert sha(path) == info['sha256'], 'Pinned public input changed: '+name
    frame = pd.read_csv(path)
    assert list(frame.columns) == ['id', 'satisfaction']
    assert len(frame) == N and frame.id.is_unique and set(frame.id) == set(ids)
    x = frame.set_index('id').loc[ids, 'satisfaction'].to_numpy(dtype=np.float64)
    assert np.isfinite(x).all() and ((x >= 0) & (x <= 1)).all()
    values[name] = x

P = np.column_stack([values[f'0.{v}.csv'] for v in ('96201','96202','96203','96204')])
teacher_c = values['0.96176.csv']
teacher_d = values['0.96183.csv']
teacher = 0.5*(teacher_c+teacher_d)
main_weights = np.array([0.1,0.2,0.3,0.4])
order_corrections = np.array([0.07,-0.01,-0.02,-0.04])

def hblend(matrix, strength=1.0):
    # Vectorized implementation of Nina's published h-blend rule.
    # Preserve its float32 intermediate sides and stable within-row tie ordering.
    sides = []
    for reverse in (False, True):
        order = np.argsort(-matrix if reverse else matrix, axis=1, kind='stable')
        side = np.zeros(N, dtype=np.float64)
        for j in range(4):
            k = order[:,j]
            side += matrix[np.arange(N),k]*(main_weights[k]+strength*order_corrections[j])
        sides.append(side.astype(np.float32))
    return (np.float32(0.3)*sides[0]+np.float32(0.7)*sides[1]).astype(np.float32)

h4 = hblend(0.7*P + 0.3*teacher_c[:,None])
h5 = hblend(0.7*P + 0.3*teacher_d[:,None])
for key, vector in [('h4',h4),('h5',h5)]:
    digest = hashlib.sha256(ranks(vector).astype('<f8').tobytes()).hexdigest()
    assert digest == REFERENCE_RANK_HASHES[key], 'Top-source ordering reproduction failed: '+key

if RECIPE == 'teacher_consensus':
    raw = hblend(0.7*P+0.3*teacher[:,None])
elif RECIPE == 'teacher40':
    raw = hblend(0.6*P+0.4*teacher[:,None])
elif RECIPE == 'half_correction':
    raw = hblend(0.7*P+0.3*teacher[:,None], strength=0.5)
elif RECIPE == 'logit_consensus':
    raw = 0.5*(logit(np.clip(h4.astype(float),1e-12,1-1e-12))+logit(np.clip(h5.astype(float),1e-12,1-1e-12)))
elif RECIPE == 'disagreement_gate':
    weight = 0.3 + 0.02*(R(P.std(axis=1))-0.5)
    raw = hblend((1-weight[:,None])*P+weight[:,None]*teacher[:,None])
elif RECIPE == 'robust_rank':
    a,b = R(h4),R(h5)
    raw = a+0.5*np.clip(b-a,-0.001,0.001)
else:
    raise ValueError(RECIPE)

prediction = R(raw)
submission = pd.DataFrame({'id':ids,'satisfaction':prediction})
submission.to_csv(OUTPUT/'submission.csv',index=False,float_format='%.15f')
saved = pd.read_csv(OUTPUT/'submission.csv')
assert np.array_equal(saved.id.to_numpy(),ids)
assert saved.satisfaction.between(0,1,inclusive='neither').all()
assert np.array_equal(ranks(saved.satisfaction),ranks(prediction))
assert np.any(prediction != R(h4)) and np.any(prediction != R(h5)), 'Unchanged ranking'
diagnostics = {'recipe':RECIPE,'rows':N,'submission_sha256':sha(OUTPUT/'submission.csv'),
               'reference_rank_hashes':REFERENCE_RANK_HASHES,
               'source_hashes':SOURCES,
               'rank_corr_h4':float(np.corrcoef(R(h4),prediction)[0,1]),
               'rank_corr_h5':float(np.corrcoef(R(h5),prediction)[0,1]),
               'changed_ranks_vs_h4':int(np.count_nonzero(prediction != R(h4))),
               'public_score':None,'cv_score':None}
(OUTPUT/'diagnostics.json').write_text(json.dumps(diagnostics,indent=2))
print(json.dumps({k:v for k,v in diagnostics.items() if k not in ['source_hashes','reference_rank_hashes']},indent=2))
display(submission.head())
